# 🔍 DeepFake & AI-Generated Image Detector
### CNN + Frequency Analysis + Grad-CAM

**قبل ما تبدأي:**
1. `Runtime → Change runtime type → T4 GPU`
2. شغّلي كل cell بالترتيب


## ✅ Step 1 — Setup Kaggle
روحي على: `kaggle.com → Settings → API → Create New Token`

هينزل ملف `kaggle.json` — ارفعيه في الـ cell دي 👇

In [3]:
import os, json
from google.colab import files

# ارفعي kaggle.json من جهازك
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install kaggle -q
!kaggle --version
print('✅ Kaggle API ready!')

Saving kaggle.json to kaggle.json
Kaggle CLI 2.0.2
✅ Kaggle API ready!


## ✅ Step 2 — Download Dataset

In [ ]:
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

!kaggle datasets download -d xhlulu/140k-real-and-fake-faces --unzip -p data/

for split in ['train','valid','test']:
    for cls in ['real','fake']:
        path = f'data/real_vs_fake/{split}/{cls}'
        if os.path.exists(path):
            print(f'{split}/{cls}: {len(os.listdir(path)):,} images')

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
 60% 2.26G/3.75G [02:08<01:13, 21.6MB/s]

## ✅ Step 3 — Imports

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, classification_report
import seaborn as sns, json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## ✅ Step 4 — Data Loaders

In [ ]:
DATA_DIR = 'data/real_vs_fake/real-vs-fake'

train_tf = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', train_tf)
val_ds   = datasets.ImageFolder(f'{DATA_DIR}/valid', val_tf)
test_ds  = datasets.ImageFolder(f'{DATA_DIR}/test',  val_tf)

print(f'Class mapping: {train_ds.class_to_idx}')
FAKE_IDX = train_ds.class_to_idx.get('fake', 0)
REAL_IDX = train_ds.class_to_idx.get('real', 1)

train_loader = DataLoader(train_ds, 64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  64, shuffle=False, num_workers=2)

print(f'Train:{len(train_ds):,}  Val:{len(val_ds):,}  Test:{len(test_ds):,}')

# عرض samples
fig, axes = plt.subplots(2,8,figsize=(20,6))
fig.suptitle('Real vs Fake Samples', fontsize=13)
for row, cls in enumerate(['real','fake']):
    cls_idx = train_ds.class_to_idx[cls]
    paths   = [p for p,l in train_ds.samples if l==cls_idx][:8]
    for col, path in enumerate(paths):
        axes[row,col].imshow(Image.open(path))
        axes[row,col].set_title(cls.upper(), fontsize=8, color='green' if cls=='real' else 'red')
        axes[row,col].axis('off')
plt.tight_layout(); plt.savefig('results/samples.png',dpi=120); plt.show()

## ✅ Step 5 — FFT Features

In [ ]:
def fft_batch(batch):
    """DeepFakes بيسيبوا signature في الـ frequency domain"""
    B,C,H,W = batch.shape
    out = torch.zeros_like(batch)
    imgs = batch.cpu().numpy()
    for i in range(B):
        for c in range(C):
            f   = np.fft.fft2(imgs[i,c])
            mag = np.log1p(np.abs(np.fft.fftshift(f))).astype('float32')
            mn,mx = mag.min(), mag.max()
            out[i,c] = torch.tensor((mag-mn)/(mx-mn+1e-8))
    return out.to(batch.device)

print('✅ FFT ready')

## ✅ Step 6 — Models

In [ ]:
class PixelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),  nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),  nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1),  nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(64,128,3,padding=1),  nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*16,512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512,128),    nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,2),
        )
    def forward(self,x): return self.classifier(self.features(x))

class FreqCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),  nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16,256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256,64),     nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64,2),
        )
    def forward(self,x): return self.classifier(self.features(x))

class Ensemble(nn.Module):
    def __init__(self,p,f):
        super().__init__()
        self.pixel=p; self.freq=f
        self.fusion=nn.Sequential(nn.Linear(4,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,2))
    def forward(self,xp,xf):
        return self.fusion(torch.cat([self.pixel(xp),self.freq(xf)],dim=1))

pixel_model = PixelCNN().to(device)
freq_model  = FreqCNN().to(device)
print(f'PixelCNN: {sum(p.numel() for p in pixel_model.parameters()):,} params')
print(f'FreqCNN:  {sum(p.numel() for p in freq_model.parameters()):,} params')

## ✅ Step 7 — Train

In [ ]:
def train_one(model, use_fft, save_path, epochs=20, name=''):
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt  = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    hist = {'tl':[],'ta':[],'vl':[],'va':[]}
    best = 0
    print(f'\n{name} — {epochs} epochs')
    print(f'{"Ep":>3} | {"TrLoss":>7} | {"TrAcc":>6} | {"VaLoss":>7} | {"VaAcc":>6}')
    print('─'*45)
    for ep in range(1, epochs+1):
        model.train(); tl=tc=tt=0
        for imgs,yb in train_loader:
            imgs,yb = imgs.to(device),yb.to(device)
            xin = fft_batch(imgs) if use_fft else imgs
            out = model(xin); loss=crit(out,yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            tl+=loss.item()*yb.size(0); tc+=(out.argmax(1)==yb).sum().item(); tt+=yb.size(0)
        sch.step()
        model.eval(); vl=vc=vt=0
        with torch.no_grad():
            for imgs,yb in val_loader:
                imgs,yb = imgs.to(device),yb.to(device)
                xin = fft_batch(imgs) if use_fft else imgs
                out = model(xin); loss=crit(out,yb)
                vl+=loss.item()*yb.size(0); vc+=(out.argmax(1)==yb).sum().item(); vt+=yb.size(0)
        ta,va_ = 100*tc/tt, 100*vc/vt
        hist['tl'].append(tl/tt); hist['ta'].append(ta)
        hist['vl'].append(vl/vt); hist['va'].append(va_)
        star = ' 💾' if va_>best else ''
        if va_>best: best=va_; torch.save(model.state_dict(),save_path)
        if ep%5==0 or ep==1:
            print(f'{ep:>3} | {tl/tt:>7.4f} | {ta:>5.2f}% | {vl/vt:>7.4f} | {va_:>5.2f}%{star}')
    print(f'✅ Best Val: {best:.2f}%')
    return hist, best

hist_pixel, best_pixel = train_one(pixel_model, False, 'models/pixel.pth', 20, '🧠 PixelCNN')
hist_freq,  best_freq  = train_one(freq_model,  True,  'models/freq.pth',  20, '〰️ FreqCNN')

## ✅ Step 8 — Ensemble

In [ ]:
pixel_model.load_state_dict(torch.load('models/pixel.pth', map_location=device))
freq_model.load_state_dict( torch.load('models/freq.pth',  map_location=device))

ensemble = Ensemble(pixel_model, freq_model).to(device)
for p in ensemble.pixel.parameters(): p.requires_grad=False
for p in ensemble.freq.parameters():  p.requires_grad=False

ens_crit = nn.CrossEntropyLoss()
ens_opt  = optim.Adam(ensemble.fusion.parameters(), lr=5e-3)
best_ens = 0

print('🚀 Ensemble (15 epochs)...')
for ep in range(15):
    ensemble.train()
    for imgs,yb in train_loader:
        imgs,yb = imgs.to(device),yb.to(device)
        loss = ens_crit(ensemble(imgs, fft_batch(imgs)), yb)
        ens_opt.zero_grad(); loss.backward(); ens_opt.step()
    ensemble.eval(); vc=vt=0
    with torch.no_grad():
        for imgs,yb in val_loader:
            imgs,yb = imgs.to(device),yb.to(device)
            out = ensemble(imgs, fft_batch(imgs))
            vc+=(out.argmax(1)==yb).sum().item(); vt+=yb.size(0)
    va_ = 100*vc/vt
    if va_>best_ens: best_ens=va_; torch.save(ensemble.state_dict(),'models/ensemble.pth')
    if (ep+1)%5==0: print(f'  Ep {ep+1}: {va_:.2f}%')
print(f'✅ Ensemble Best: {best_ens:.2f}%')

## ✅ Step 9 — Evaluation & Charts

In [ ]:
ensemble.load_state_dict(torch.load('models/ensemble.pth', map_location=device))
ensemble.eval()
preds=[]; probs=[]; labels=[]
with torch.no_grad():
    for imgs,yb in test_loader:
        imgs = imgs.to(device)
        out  = ensemble(imgs, fft_batch(imgs))
        pr   = torch.softmax(out,dim=1)
        preds.extend(out.argmax(1).cpu().numpy())
        probs.extend(pr[:,FAKE_IDX].cpu().numpy())
        labels.extend(yb.numpy())

preds=np.array(preds); probs=np.array(probs); labels=np.array(labels)
acc = (preds==labels).mean()*100
auc = roc_auc_score(labels,probs)
print(f'🎯 Test Accuracy: {acc:.2f}%')
print(f'📈 AUC-ROC:       {auc:.4f}')
print(classification_report(labels,preds,target_names=['REAL','FAKE']))

fig,axes=plt.subplots(2,3,figsize=(18,10))
fig.suptitle('DeepFake Detector — Results',fontsize=14,fontweight='bold')
for row,(hist,name) in enumerate([(hist_pixel,'PixelCNN'),(hist_freq,'FreqCNN')]):
    eps=range(1,len(hist['tl'])+1)
    axes[row,0].plot(eps,hist['tl'],'b-o',ms=3,label='Train'); axes[row,0].plot(eps,hist['vl'],'r-o',ms=3,label='Val')
    axes[row,0].set_title(f'{name} Loss'); axes[row,0].legend(); axes[row,0].grid(alpha=.3)
    axes[row,1].plot(eps,hist['ta'],'b-o',ms=3,label='Train'); axes[row,1].plot(eps,hist['va'],'r-o',ms=3,label='Val')
    axes[row,1].set_title(f'{name} Accuracy'); axes[row,1].legend(); axes[row,1].grid(alpha=.3)
fpr,tpr,_=roc_curve(labels,probs)
axes[0,2].plot(fpr,tpr,'b-',lw=2,label=f'AUC={auc:.3f}'); axes[0,2].plot([0,1],[0,1],'r--',alpha=.5)
axes[0,2].set_title('ROC Curve'); axes[0,2].legend(); axes[0,2].grid(alpha=.3)
cm=confusion_matrix(labels,preds)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=axes[1,2],xticklabels=['REAL','FAKE'],yticklabels=['REAL','FAKE'])
axes[1,2].set_title('Confusion Matrix')
plt.tight_layout(); plt.savefig('results/training.png',dpi=140); plt.show()

np.save('data/norm_mean.npy', np.array([0.485,0.456,0.406],'float32').reshape(3,1,1))
np.save('data/norm_std.npy',  np.array([0.229,0.224,0.225],'float32').reshape(3,1,1))
json.dump(dict(accuracy=round(acc,4),auc=round(auc,4),pixel_best=round(best_pixel,4),
               freq_best=round(best_freq,4),ensemble_best=round(best_ens,4),fake_idx=FAKE_IDX),
          open('results/results.json','w'),indent=2)
print(f'🎉 Done! Accuracy:{acc:.2f}%  AUC:{auc:.4f}')

## ✅ Step 10 — Download Models

In [ ]:
from google.colab import files
for f in ['models/ensemble.pth','models/pixel.pth','models/freq.pth',
          'data/norm_mean.npy','data/norm_std.npy','results/results.json']:
    files.download(f)
    print(f'📥 {f}')
print('\n✅ حطيهم في app/ وشغّلي: streamlit run app.py')